# VeloCity × Jev: Schadenmeldungen beurteilen

Kundinnen und Kunden von VeloCity Würzburg melden Schäden in freier Sprache. Jev, das System-One-Modell
von TypeSafe, beantwortet zu jeder Meldung fünf Fragen mit Wahrscheinlichkeiten. Welche Entscheidung
daraus folgt (sperren, prüfen, Wartungsauftrag, kein Schaden), legt der Code in `velocity_jev/regeln.py`
über Schwellen fest.

**Aufbau des Notebooks**

1. Einrichtung: Pakete, Projektordner, API-Key
2. Parameter: Fragen-Stand, Datensatz, Cache
3. Die fünf Fragen an Jev
4. Meldungen beurteilen
5. Kennzahlen
6. Alle Meldungen als interaktive Tabelle
7. Gruppierte Auswertung
8. Fehlentscheidungen und Kreuztabellen
9. Schwellen durchspielen
10. Fehleranalyse des Ausgangsstands
11. Vergleich der Fragen-Stände
12. Eine einzelne Meldung live beurteilen
13. Ergebnisse nach Supabase schreiben und in der Warenwirtschaft zeigen
14. Accuracy, Precision, Recall und F1, Konfusionsmatrizen
15. Vergleich mit einem Encoder-Modell aus der BERT-Familie
16. Vergleich mit einem trainierten Klassifikator

Alle Antworten von Jev liegen im Cache `daten/cache/jev_cache.jsonl`, die des Encoder-Modells in
`daten/cache/encoder_nli.csv`. Ohne API-Key lässt sich das Notebook vollständig ausführen, solange die
Fragen nicht umformuliert werden.

## 1 Einrichtung

**API-Key in Deepnote hinterlegen** (nur nötig, wenn neue Anfragen an Jev gehen sollen):

1. Rechte Seitenleiste → **Integrations** → **+ Add integration** → **Environment variables**.
2. Name `TYPESAFE_API_KEY`, als Wert den Key aus der TypeSafe-Konsole eintragen, speichern.
3. Die Integration mit diesem Projekt verbinden (**Connect**) und die Maschine neu starten.

Deepnote legt den Wert verschlüsselt ab und stellt ihn als Umgebungsvariable bereit. Der Key gehört
nie in eine Notebook-Zelle, weil Zellen beim Teilen des Projekts sichtbar sind.

**Verbindung zur Datenbank** (nur für Abschnitt 13, nur im Projekt der Lehrperson): in derselben
Integration eine zweite Variable `DATABASE_URL` anlegen. Der Wert steht in `jev/.env` und verbindet sich
als Rolle `jev_schreiber`, die nur in das Schema `jev_labor` einfügen darf.

In [ ]:
import hashlib
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_DEEPNOTE = "DEEPNOTE_PROJECT_ID" in os.environ


def paket_archive(ordner: Path) -> list[Path]:
    """ZIP-Dateien im Ordner, die das Paket velocity_jev enthalten, die neueste zuletzt."""
    archive = []
    for archiv in ordner.glob("*.zip"):
        with zipfile.ZipFile(archiv) as z:
            if any(name.startswith("velocity_jev/") for name in z.namelist()):
                archive.append(archiv)
    return sorted(archive, key=lambda a: a.stat().st_mtime)


def projektordner_finden() -> Path:
    """Sucht den Ordner mit dem Paket velocity_jev. In Deepnote wird eine neu hochgeladene ZIP-Datei entpackt."""
    start = Path.cwd()
    archive = paket_archive(start)
    if archive and (IN_DEEPNOTE or not (start / "velocity_jev").is_dir()):
        marke = start / ".paket_stand"
        stand = hashlib.sha256(archive[-1].read_bytes()).hexdigest()
        if not (start / "velocity_jev").is_dir() or not marke.exists() or marke.read_text() != stand:
            with zipfile.ZipFile(archive[-1]) as z:
                z.extractall(start)
            marke.write_text(stand)
            print(f"{archive[-1].name} entpackt.")
        return start
    for ordner in [start, *[p for p in start.iterdir() if p.is_dir()]]:
        if (ordner / "velocity_jev").is_dir():
            return ordner
    raise FileNotFoundError("Ordner velocity_jev nicht gefunden. Bitte die Projektdateien hochladen.")


def pakete_installieren(*argumente: str) -> None:
    """pip nur in Deepnote oder in einer virtuellen Umgebung, damit es keine gemeinsam genutzte Installation verstellt."""
    if not IN_DEEPNOTE and sys.prefix == sys.base_prefix:
        raise ModuleNotFoundError(
            "Pakete fehlen. Bitte in einer virtuellen Umgebung installieren: "
            "python3 -m venv .venv && .venv/bin/pip install -r requirements.txt"
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *argumente], check=True)


PROJEKT = projektordner_finden()
sys.path.insert(0, str(PROJEKT))
print("Projektordner:", PROJEKT)
print("Python:", sys.version.split()[0])
if sys.version_info < (3, 10):
    print("typesafe-sdk braucht Python 3.10 oder neuer: in Deepnote unter Environment eine neuere Version wählen.")

try:
    import matplotlib  # noqa: F401
    import sklearn  # noqa: F401
    import typesafe_sdk  # noqa: F401
except ImportError:
    print("Installiere Pakete aus requirements.txt ...")
    pakete_installieren("-r", str(PROJEKT / "requirements.txt"))

In [ ]:
import pandas as pd
from IPython.display import display
from typesafe_sdk import TypeSafeClient

from velocity_jev.ablauf import klassifizieren
from velocity_jev.auswertung import (
    KOSTEN, STUFEN, alle_kennzahlen, gruppiert, kennzahlen_tabelle, schwellen_durchspielen, vergleich_bauen,
)
from velocity_jev.fragen import AKTUELLER_STAND, STAENDE, fragen
from velocity_jev.konfig import DATENSAETZE, ERGEBNISSE, MODELL, api_key_vorhanden
from velocity_jev.pipeline import Cache, beurteilen
from velocity_jev.protokoll import auswertung_ablegen, lauf_ablegen
from velocity_jev.regeln import PRUEFEN, entscheiden

pd.set_option("display.max_colwidth", 120)
print("Modell:", MODELL)
print("API-Key:", "gesetzt" if api_key_vorhanden() else "nicht gesetzt (nur Cache)")

## 2 Parameter

- `STAND`: Wortlaut der Fragen. Stand 0 ist die Übergabe, Stand 1 der beste Stand, Stand 2 wurde
  verworfen. Seit der Regel für sicherheitsrelevante Meldungen ohne Schaden ist Stand 2 auf den
  48 Meldungen günstiger; auf dem Holdout entscheiden Stand 1 und 2 gleich (siehe `docs/iterationen.md`).
- `DATENSATZ`: `"meldungen"` (48 Meldungen, an denen die Fragen verbessert wurden) oder `"holdout"`
  (18 Meldungen, die erst nach der Wahl des besten Stands gelaufen sind).
- `NUR_CACHE`: `True` liest nur gespeicherte Antworten und verbraucht keine Tokens. Bei `False` gehen
  Meldungen, die noch nicht im Cache liegen, an Jev; dafür ist der API-Key nötig.

In [ ]:
STAND = AKTUELLER_STAND
DATENSATZ = "meldungen"
NUR_CACHE = True

for nummer, beschreibung in STAENDE.items():
    print(f"Stand {nummer}: {beschreibung}{'   ← gewählt' if nummer == STAND else ''}")

## 3 Die fünf Fragen an Jev

Alle fünf Fragen gehen in einem Request an Jev und beziehen sich auf denselben State: den Meldungstext
und den Radtyp. Jev liest die Fragen wörtlich; was zur Erklärung eines Fehlers nötig wäre, gehört in
`instructions` oder `criteria`.

In [ ]:
def fragen_tabelle(stand: int) -> pd.DataFrame:
    """Die Fragen eines Stands mit Primitive, Anweisung und Kriterien als Tabelle."""
    zeilen = []
    for name, frage in fragen(stand).items():
        daten = frage.model_dump(mode="json")
        kriterien = daten.get("criteria")
        if isinstance(kriterien, dict):
            kriterien = " | ".join(f"{k}: {v}" for k, v in kriterien.items())
        elif isinstance(kriterien, list):
            kriterien = " | ".join(kriterien)
        zeilen.append({"frage": name, "primitive": type(frage).__name__,
                       "instructions": daten["instructions"], "criteria": kriterien or "–"})
    return pd.DataFrame(zeilen)


fragen_tabelle(STAND)

## 4 Meldungen beurteilen

Jede Meldung wird einmal an Jev geschickt oder aus dem Cache gelesen; danach wendet `regeln.py` die
Schwellen an. Jeder Lauf wird unter `ergebnisse/laeufe/<lauf_id>/` abgelegt und in
`ergebnisse/laeufe.csv` eingetragen.

In [ ]:
meldungen = pd.read_csv(DATENSAETZE[DATENSATZ], dtype={"meldung_id": str})

client = None
if not NUR_CACHE:
    if not api_key_vorhanden():
        raise RuntimeError("TYPESAFE_API_KEY fehlt, siehe Abschnitt 1. Mit NUR_CACHE = True geht es ohne Key.")
    client = TypeSafeClient(model=MODELL)

urteile, lauf = klassifizieren(meldungen, client=client, modell=MODELL, stand=STAND,
                               datensatz=DATENSATZ, anmerkung="Notebook", ausgabe=False)
if client is not None:
    client.close()
lauf_ablegen(ERGEBNISSE, lauf, urteile)

print(f"Lauf {lauf['lauf_id']}: {lauf['meldungen']} Meldungen beurteilt, {lauf['neue_requests']} neu an Jev")
print(f"Tokens dieses Laufs: {lauf['input_tokens']} Input, {lauf['output_tokens']} Output")
print(f"Tokens aller Urteile (auch aus dem Cache): {lauf['input_tokens_alle_urteile']} Input, "
      f"{lauf['output_tokens_alle_urteile']} Output")
if lauf["ohne_urteil"]:
    print("Ohne Urteil, weil nicht im Cache:", lauf["ohne_urteil"])

## 5 Kennzahlen

Die Soll-Entscheidung folgt aus den Labels in der CSV-Datei: kein Schaden → weiterleiten;
sicherheitsrelevant oder fahruntauglich → sperren; Schaden ohne passende Kategorie → prüfen;
sonst → Wartungsauftrag. Die Kosten beruhen auf groben Annahmen, die in `velocity_jev/auswertung.py`
stehen.

In [ ]:
vergleich = vergleich_bauen(meldungen, urteile)
kennzahlen = alle_kennzahlen(vergleich)
auswertung_ablegen(ERGEBNISSE, lauf, vergleich, kennzahlen)

print("Kostenannahmen je Fall:", KOSTEN)
kennzahlen_tabelle(kennzahlen).round(3)

## 6 Alle Meldungen

Eine Zeile je Meldung mit Soll und Ist. In Deepnote lässt sich die Tabelle über die Spaltenköpfe
sortieren und filtern; unter **Cell formatting** kann man etwa `sicherheitsrelevant` als Farbskala
einfärben. Die Spalte `ergebnis` fasst zusammen, ob die Entscheidung richtig war und welche Art von
Fehler vorliegt.

In [ ]:
SPALTEN = [
    "meldung_id", "merkmal", "typ_code", "text", "ergebnis", "soll_entscheidung", "entscheidung", "begruendung",
    "soll_kategorie", "kategorie", "kategorie_konfidenz", "soll_schwere", "schwere_stufe", "schwere_score",
    "schwere_konfidenz", "soll_ist_schaden", "ist_schadensmeldung", "soll_sicherheitsrelevant",
    "sicherheitsrelevant", "soll_personenschaden", "personenschaden", "eskalation",
]
tabelle = vergleich[SPALTEN].round(2)
tabelle

## 7 Gruppierte Auswertung

Die Deepnote-Tabelle sortiert und filtert, gruppiert aber nicht. Die Gruppierung übernimmt deshalb
pandas. `GRUPPIEREN_NACH` lässt sich auf jede Spalte der Tabelle aus Abschnitt 6 setzen, sinnvoll sind
etwa `"merkmal"`, `"ergebnis"`, `"soll_kategorie"`, `"typ_code"` oder `"soll_entscheidung"`.
Ohne Code geht es in Deepnote auch mit einem **Pivot table**-Block auf die Variable `tabelle`.

In [ ]:
GRUPPIEREN_NACH = "merkmal"
gruppiert(vergleich, GRUPPIEREN_NACH)

## 8 Fehlentscheidungen und Kreuztabellen

Zuerst alle Meldungen, bei denen der Code anders entschieden hat als das Soll, danach die
Kreuztabellen der Entscheidung und der Schwere (Zeilen: Soll, Spalten: Ergebnis).

In [ ]:
abweichungen = tabelle[tabelle.entscheidung != tabelle.soll_entscheidung]
print(f"{len(abweichungen)} von {len(tabelle)} Entscheidungen weichen vom Soll ab.")
abweichungen[["meldung_id", "merkmal", "ergebnis", "soll_entscheidung", "entscheidung", "begruendung",
              "sicherheitsrelevant", "ist_schadensmeldung", "schwere_stufe", "text"]]

In [ ]:
entscheidungen = pd.crosstab(vergleich.soll_entscheidung, vergleich.entscheidung,
                             rownames=["Soll"], colnames=["Code"])
schaden = vergleich[vergleich.soll_ist_schaden]
reihenfolge = list(STUFEN)
schwere = (pd.crosstab(schaden.soll_schwere, schaden.schwere_stufe, rownames=["Soll"], colnames=["Jev"])
           .reindex(index=reihenfolge, columns=reihenfolge, fill_value=0))
display(entscheidungen)
display(schwere)

## 9 Schwellen durchspielen

Die Wahrscheinlichkeiten ändern sich nicht, wenn man Schwellen verschiebt; deshalb kostet diese
Rechnung keine Tokens. Die Tabelle ist nach Kosten sortiert. Eine Schwelle, die nur auf diesen
Meldungen optimal ist, gilt nicht automatisch für neue Meldungen.

In [ ]:
schwellen_durchspielen(vergleich).round(2)

## 10 Fehleranalyse des Ausgangsstands

Für Stand 0 wurde jede Fehlentscheidung einer von vier Ursachen zugeordnet: Frage unklar, Label
strittig, bekannte Grenze von Jev, Fehler im Code. Die Zuordnung ist eine Einschätzung, keine Messung.

In [ ]:
fehleranalyse = pd.read_csv(PROJEKT / "docs" / "fehleranalyse_stand0.csv", dtype={"meldung_id": str})
display(pd.crosstab(fehleranalyse.merkmal, fehleranalyse.ursache, margins=True, margins_name="Summe"))
fehleranalyse

## 11 Vergleich der Fragen-Stände

Jeder Stand und jeder Datensatz wird aus dem Cache ausgewertet; es entstehen keine Kosten.
Die Kennzahlen stehen je Lauf nebeneinander.

In [ ]:
def stand_auswerten(stand: int, datensatz: str) -> dict:
    """Einen Fragen-Stand auf einem Datensatz nur aus dem Cache auswerten."""
    daten = pd.read_csv(DATENSAETZE[datensatz], dtype={"meldung_id": str})
    u, l = klassifizieren(daten, client=None, modell=MODELL, stand=stand, datensatz=datensatz, ausgabe=False)
    if len(u) < len(daten):
        return {"datensatz": datensatz, "stand": stand, "hinweis": f"{len(daten) - len(u)} Meldungen nicht im Cache"}
    k = alle_kennzahlen(vergleich_bauen(daten, u))
    return {"datensatz": datensatz, "stand": stand, "beschreibung": STAENDE[stand],
            "tokens_input": l["input_tokens_alle_urteile"], **k}


vergleich_staende = pd.DataFrame([stand_auswerten(s, d) for d in DATENSAETZE for s in STAENDE])
vergleich_staende.round(3)

## 12 Eine einzelne Meldung live beurteilen

Braucht den API-Key. Ein neuer Text kostet einen Request mit etwa 1 400 Input-Tokens; derselbe Text
kommt beim zweiten Mal aus dem Cache.

In [ ]:
TEXT = "Bremse hinten quietscht und greift erst ganz spät"
RADTYP = "CITY"   # CITY, EBIKE oder CARGO

if api_key_vorhanden():
    with TypeSafeClient(model=MODELL) as c:
        u = beurteilen("LIVE", TEXT, RADTYP, client=c, modell=MODELL, cache=Cache(), stand=STAND)
    e = entscheiden(u)
    display(pd.DataFrame([
        {"Urteil": "ist_schadensmeldung", "Wert": u.ist_schadensmeldung},
        {"Urteil": "sicherheitsrelevant", "Wert": u.sicherheitsrelevant},
        {"Urteil": "personenschaden", "Wert": u.personenschaden},
        {"Urteil": f"Kategorie {u.kategorie} (Konfidenz)", "Wert": u.kategorie_konfidenz},
        {"Urteil": f"Schwere {u.schwere_stufe} (Erwartungswert)", "Wert": u.schwere_score},
    ]).round(2))
    print(f"Entscheidung: {e.entscheidung}{' + Eskalation an den Kundendienst' if e.eskalation else ''}")
    print(f"Begründung: {e.begruendung}")
    print("Aus dem Cache." if u.aus_cache else f"Neuer Request: {u.input_tokens} Input-, {u.output_tokens} Output-Tokens.")
else:
    print("Kein API-Key gesetzt, siehe Abschnitt 1.")

## 13 Ergebnisse nach Supabase schreiben

Der Lauf aus Abschnitt 4 wird mit der Rolle `jev_schreiber` in das Schema `jev_labor` geschrieben:
Meldungen, Lauf und Urteile in einer Transaktion. Ein Lauf, der schon in der Datenbank steht, wird
nicht doppelt geschrieben. In der Warenwirtschaft (Instandhaltung, Reiter „Meldungseingang“) erscheint
nur ein **freigegebener** Lauf, und zwar der zuletzt freigegebene. Dort übernimmt oder verwirft die
Werkstatt jeden Vorschlag; erst die Übernahme legt eine Schadensmeldung an.

Beide Schalter stehen auf `False`. Eine Freigabe ändert, was die Werkstatt sieht, und bleibt deshalb
eine bewusste Entscheidung.

In [ ]:
SCHREIBEN = False   # True: den Lauf aus Abschnitt 4 nach Supabase schreiben
FREIGEBEN = False   # True: den Lauf zusätzlich in der Warenwirtschaft zeigen

if SCHREIBEN:
    url = os.environ.get("DATABASE_URL", "")
    if not url or "USER:PASSWORT" in url:
        raise RuntimeError("DATABASE_URL fehlt, siehe Abschnitt 1.")
    from velocity_jev.datenbank import lauf_hochladen
    ergebnis = lauf_hochladen(url, lauf, urteile, meldungen, freigeben=FREIGEBEN)
    stand = "neu geschrieben" if ergebnis["neu"] else "war schon vorhanden"
    print(f"Lauf {ergebnis['lauf_id']} {stand}, {ergebnis['urteile']} Urteile"
          f"{', freigegeben' if ergebnis['freigegeben'] else ''}.")
else:
    print("Nichts geschrieben. SCHREIBEN = True setzen, um den Lauf nach Supabase zu übertragen.")

## 14 Accuracy, Precision, Recall und F1

Abschnitt 5 zählt richtige Entscheidungen und Kosten. Die klassischen Kennzahlen der Klassifikation
messen jedes Urteil einzeln:

- **Accuracy**: Anteil der richtigen Antworten.
- **Precision**: Von den Meldungen, die das Modell einer Klasse zuordnet, wie viele gehören laut Soll dazu?
- **Recall**: Von den Meldungen, die laut Soll zu einer Klasse gehören, wie viele erkennt das Modell?
- **F1**: harmonisches Mittel aus Precision und Recall.

Bei den drei Ja/Nein-Fragen ist „ja“ die gesuchte Klasse, gemessen bei der Schwelle 0,5;
`sicherheitsrelevant` zählt wie in Abschnitt 5 nur an echten Schadensmeldungen. Für Kategorie, Schwere
und Entscheidung steht das Makro-Mittel über die Klassen, die im Soll vorkommen: Jede Klasse zählt gleich
viel, auch eine seltene. Bei `sicherheitsrelevant` ist der Recall die wichtigste Zahl, denn jede
übersehene Gefahr bleibt als Rad im Verleih.

In [ ]:
import matplotlib.pyplot as plt

from velocity_jev.evaluation import (
    entscheidungen_vergleichen, kennzahlen_je_urteil, klassen_kennzahlen, konfusionsmatrizen,
    matrizen_zeichnen, modelle_vergleichen,
)
from velocity_jev.regeln import ENTSCHEIDUNGEN

kennzahlen_je_urteil(vergleich).round(2)

Die Entscheidung des Codes je Klasse: `soll` zählt, wie oft die Klasse laut Soll vorkommt,
`vorhergesagt`, wie oft der Code sie wählt. Die Precision von `pruefen` ist niedrig, weil unsichere
Fälle absichtlich zu einem Menschen gehen, auch wenn das Soll eine andere Entscheidung vorsieht.

In [ ]:
klassen_kennzahlen(vergleich.soll_entscheidung, vergleich.entscheidung, ENTSCHEIDUNGEN).round(2)

### Konfusionsmatrizen

Zeilen: Soll, Spalten: Vorhersage des Modells beziehungsweise Entscheidung des Codes. Auf der Diagonale
stehen die richtigen Fälle, daneben die Verwechslungen. Bei der Entscheidung ist die Zeile `sperren` die
wichtigste: Ein Fall in den Spalten `auftrag` oder `kein_schaden_weiterleiten` wäre ein übersehener
Sicherheitsschaden.

In [ ]:
matrizen = konfusionsmatrizen(vergleich)
matrizen_zeichnen(matrizen, ["ist_schadensmeldung", "sicherheitsrelevant", "personenschaden", "schwere",
                             "entscheidung"])
plt.show()

In [ ]:
matrizen_zeichnen(matrizen, ["kategorie"], spalten=1, groesse=(7.5, 6.5))
plt.show()

## 15 Vergleich mit einem Encoder-Modell aus der BERT-Familie

Ein Encoder wie BERT liest einen Text vollständig und bildet daraus eine Zahlendarstellung; eine
Klassifikationsschicht darüber entscheidet. Ohne eigene Trainingsdaten lässt er sich über **Natural
Language Inference** (NLI) einsetzen: Ein darauf trainiertes Modell schätzt, ob eine Hypothese aus einem
Text folgt, ihm widerspricht oder nichts mit ihm zu tun hat. Jede Frage an Jev wird dafür in Hypothesen
übersetzt, bei den Ja/Nein-Fragen je eine für „ja“ und eine für „nein“, wie die `criteria` von Jev.

Verglichen wird `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` von Hugging Face (Lizenz
MIT, Stand 29.09.2026): 279 Mio. Parameter, von Microsoft auf Texten in 100 Sprachen vortrainiert und auf
über 2,7 Mio. NLI-Satzpaaren in 27 Sprachen feinjustiert, darunter Deutsch. Wie Jev sieht es keine
VeloCity-Meldung mit Label. Aus seinen Wahrscheinlichkeiten entstehen Urteile in derselben Form wie bei
Jev (`velocity_jev/rohwerte.py`), und derselbe Code aus `regeln.py` entscheidet mit denselben Schwellen.
Als Konfidenz gilt die höchste Wahrscheinlichkeit einer Verteilung; das ist nicht dieselbe Größe wie die
Konfidenz von Jev.

Die Wahrscheinlichkeiten liegen in `daten/cache/encoder_nli.csv`. Mit `ENCODER_NEU_RECHNEN = True`
rechnet das Notebook sie neu. Dafür lädt es das Modell (rund 1,1 GB) und braucht auf einer CPU einige
Minuten; fehlen `torch` und `transformers`, installiert es sie vorher.

In [ ]:
ENCODER_NEU_RECHNEN = False   # True: Modell laden und neu rechnen (rund 1,1 GB, einige Minuten)

if ENCODER_NEU_RECHNEN:
    try:
        import torch  # noqa: F401
        import transformers  # noqa: F401
    except ImportError:
        pakete_installieren("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
        pakete_installieren("transformers", "sentencepiece", "protobuf")

from velocity_jev.encoder import HYPOTHESEN, KATEGORIE_SATZTEILE, SCHWERE_HYPOTHESEN, VORLAGE_KATEGORIE, encoder_urteile

daten = {d: pd.read_csv(pfad, dtype={"meldung_id": str}) for d, pfad in DATENSAETZE.items()}
encoder = {d: encoder_urteile(daten[d], d, neu_rechnen=ENCODER_NEU_RECHNEN) for d in DATENSAETZE}

display(pd.DataFrame([{"frage": f, "hypothese ja": h["ja"], "hypothese nein": h["nein"]}
                      for f, h in HYPOTHESEN.items()]))
display(pd.DataFrame({"schwere": list(SCHWERE_HYPOTHESEN), "hypothese": list(SCHWERE_HYPOTHESEN.values())}))
print("Kategorie:", VORLAGE_KATEGORIE.format(KATEGORIE_SATZTEILE["Bremse"]), "… eine Hypothese je Kategorie")

In [ ]:
def jev_urteile(datensatz: str) -> pd.DataFrame:
    """Die Urteile von Jev zum gewählten Stand, nur aus dem Cache."""
    urteile_jev, _ = klassifizieren(daten[datensatz], client=None, modell=MODELL, stand=STAND,
                                    datensatz=datensatz, ausgabe=False)
    return urteile_jev


vergleiche = {d: {"Jev": vergleich_bauen(daten[d], jev_urteile(d)),
                  "Encoder (NLI)": vergleich_bauen(daten[d], encoder[d])} for d in DATENSAETZE}
modelle_vergleichen(vergleiche["meldungen"]).round(2)

In [ ]:
entscheidungen_vergleichen(vergleiche["meldungen"]).round(2)

In [ ]:
matrizen_zeichnen(konfusionsmatrizen(vergleiche["meldungen"]["Encoder (NLI)"]),
                  ["sicherheitsrelevant", "personenschaden", "schwere", "entscheidung"], spalten=4)
plt.show()

**Beobachtungen** (48 Meldungen, Jev mit Fragen-Stand 1, Encoder aus dem Cache vom 29.09.2026)

- Ob überhaupt ein Schaden vorliegt, erkennen beide ähnlich gut: Accuracy 96 % bei Jev, 92 % beim Encoder.
- Die Schwere stuft der Encoder bei allen 48 Meldungen als `mittel` ein. Seine Wahrscheinlichkeiten
  verteilen sich so gleichmäßig, dass der Erwartungswert immer in der Mitte liegt.
- Bei `personenschaden` liegt er in 43 von 48 Meldungen über 0,5; laut Soll trifft es auf 3 zu. Eine
  Meldung ohne Angaben zu Personen passt für das Modell eher zu „gestürzt“ als zu „unverletzt“.
- Die Kategorie trifft er bei 17 von 42 Schadensmeldungen. Seine höchste Wahrscheinlichkeit liegt im
  Median bei 0,34, die Regeln verlangen 0,5; viele Meldungen gehen deshalb zur Prüfung.
- Übersehen wird kein Sicherheitsschaden: `sicherheitsrelevant` liegt beim Encoder in 18 von 48
  Meldungen bei mindestens 0,8. Dafür sperrt er 7 Räder unnötig.

Die erste Fassung hatte je Ja/Nein-Frage nur eine Hypothese. Mit Gegenhypothesen stieg der Anteil
richtiger Entscheidungen von 25 % auf 33 %; bei `personenschaden` half es nicht. Weiter angepasst wurden
die Hypothesen nicht, damit der Vergleich nicht auf diese 48 Meldungen zugeschnitten ist.

## 16 Vergleich mit einem trainierten Klassifikator

Vor den Sprachmodellen lernte man je Frage einen Klassifikator aus Beispielen mit Label. Hier:
Zeichen-n-Gramme aus drei bis fünf Zeichen, gewichtet mit TF-IDF, und je Frage eine logistische
Regression (`velocity_jev/klassisch.py`). Trainiert wird auf den 48 Meldungen, geprüft auf den
18 Holdout-Meldungen, die das Modell nicht gesehen hat; auf den 48 Meldungen selbst wären die
Kennzahlen geschönt. Jev und der Encoder urteilen auch auf dem Holdout ohne Training.

18 Meldungen sind wenig: Eine Meldung mehr oder weniger richtig verschiebt die Accuracy um rund
6 Prozentpunkte. Die Soll-Labels des Holdouts sind noch nicht von einer zweiten Person geprüft.

In [ ]:
from velocity_jev.klassisch import klassisch_urteile

holdout = vergleiche["holdout"]
holdout["TF-IDF + LR"] = vergleich_bauen(daten["holdout"], klassisch_urteile(daten["meldungen"], daten["holdout"]))
modelle_vergleichen(holdout).round(2)

In [ ]:
entscheidungen_vergleichen(holdout).round(2)

**Beobachtungen** (18 Holdout-Meldungen, davon 15 Schadensmeldungen)

- Jev entscheidet 67 % richtig, der Encoder 39 %, der trainierte Klassifikator 6 %. Keines der drei
  Modelle übersieht einen Sicherheitsschaden.
- Der trainierte Klassifikator schickt alle 18 Meldungen zur Prüfung. Bei 16 Kategorien und 48
  Beispielen liegt seine höchste Wahrscheinlichkeit für eine Kategorie nie über 0,14; die Regeln
  verlangen 0,5.
- Bei `sicherheitsrelevant` liegt er vorn: F1 0,88 gegenüber 0,75 bei Jev, gemessen an 15
  Schadensmeldungen.

### Offen: Vergleich mit einem generativen LLM

Ein generatives Sprachmodell mit Prompt und JSON-Ausgabe ließe sich genauso einbinden: Seine Antworten in
die Form von `velocity_jev/rohwerte.py` bringen, dann entscheidet derselbe Code. Dafür ist ein API-Key des
Anbieters nötig; der Vergleich ist deshalb noch nicht gerechnet.